# 🛰️ Planet Orders API → Google Cloud Storage → GEE Pipeline
**Mining Site Land Use Change Analysis**

### Workflow
1. Install dependencies
2. Configure credentials & AOI
3. Search Planet scenes
4. Place order (Planet delivers to GCS)
5. Monitor order status
6. Ingest from GCS into GEE
7. Save GEE visualisation script

> ⚠️ Run cells **in order**. Each section is independent so you can re-run individual steps if needed.

---
## Cell 1 — Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install planet earthengine-api google-cloud-storage nest_asyncio --quiet
print('✅ All packages installed')

✅ All packages installed


---
## Cell 2 — Imports
> `nest_asyncio` is required to run async code inside Jupyter

In [2]:
import os
import asyncio
import json
import time
from pathlib import Path
from datetime import datetime

import nest_asyncio
nest_asyncio.apply()  # allows asyncio.run() inside Jupyter

import planet
from google.cloud import storage
import ee

print('✅ Imports successful')

✅ Imports successful


---
## Cell 3 — ⚙️ Configuration
> **Edit all values marked with `← CHANGE` before running anything else**

In [7]:
# ── Planet ───────────────────────────────────────────────────────────────
# PLANET_API_KEY  = 'YOUR_PLANET_API_KEY'    

from dotenv import load_dotenv

load_dotenv(r"D:\natcap\mongolia-mining\.env")

API_KEY = os.getenv("PLANET_API_KEY")
print("Key loaded:", API_KEY is not None)  # confirm without exposing key

# if your Planet API Key is not set as an environment variable, you can paste it below
if 'PLANET_API_KEY' in os.environ:
    API_KEY = os.environ['PLANET_API_KEY']
    print("Key loaded")
else:
    print("Key not found in environment variables, using hard coded key (not recommended)")

Key loaded: True
Key loaded


In [ ]:

# ── Area of Interest (bounding box [west, south, east, north]) ───────────

CENTER_LON  =  104.5350414;  
CENTER_LAT  =   48.4000713;  
AOI_GEOJSON = {
    "type": "Polygon",
    "coordinates": [[
        [CENTER_LON-0.2, CENTER_LAT-0.2],   # [lon, lat] bottom-left
        [CENTER_LON+0.2, CENTER_LAT+0.2],   # bottom-right
        [CENTER_LON+0.2,  CENTER_LAT+0.2],   # top-right
        [CENTER_LON-0.2,  CENTER_LAT+0.2],   # top-left
        [CENTER_LON-0.2, CENTER_LAT-0.2]    # close ring
    ]]
}


# ── Date range ───────────────────────────────────────────────────────────
START_DATE = '2015-01-01'
END_DATE   = '2024-01-01'

# ── Planet scene settings ────────────────────────────────────────────────
ITEM_TYPE   = 'PSScene'                 # PlanetScope 3m daily
ASSET_TYPE  = 'ortho_analytic_4b_sr'    # 4-band surface reflectance
BUNDLE_TYPE = 'analytic_sr_udm2'        # SR + UDM2 cloud mask
MAX_CLOUD   = 0.1                       # 10% max cloud cover
MAX_SCENES  = 5                         # max scenes per order

# ── Google Cloud Storage ─────────────────────────────────────────────────
GCS_BUCKET      = 'planet-mining'                 # ← CHANGE 'your-gcs-bucket-name'
GCS_PREFIX      = 'planet-mining-site'            # folder inside bucket
GCS_CREDENTIALS = r'D:\natcap\mongolia-mining\code\credentials\planet-mining-490623-d1415fceccb2.json'  # ← CHANGE (or None for ADC) 'path/to/service-account.json'

# ── Google Earth Engine ──────────────────────────────────────────────────
GEE_PROJECT     = 'gee-planet-natcap'        # ← CHANGE 'your-gee-project-id'
GEE_ASSET_PATH  = f'projects/{GEE_PROJECT}/assets/planet_mining'

# ── Local staging ────────────────────────────────────────────────────────
LOCAL_DIR = Path('./planet_downloads')
LOCAL_DIR.mkdir(exist_ok=True)

print('✅ Configuration loaded')
print(f'   AOI: {AOI_GEOJSON["coordinates"][0][0]} → {AOI_GEOJSON["coordinates"][0][2]}')
print(f'   Date range: {START_DATE} → {END_DATE}')
print(f'   GCS bucket: {GCS_BUCKET}')
print(f'   GEE asset:  {GEE_ASSET_PATH}')

---
## Cell 4 — 🔐 Authenticate GEE

In [4]:
# Run this once — opens a browser window to authenticate
# ee.Authenticate()
# ee.Authenticate(auth_mode='notebook')

ee.Initialize(project=GEE_PROJECT)
print('✅ GEE authenticated and initialised')

✅ GEE authenticated and initialised


---
## Cell 5 — 🔍 Search Planet Scenes

In [5]:
async def search_scenes():
    async with planet.Session(auth=planet.Auth.from_key(PLANET_API_KEY)) as sess:
        client = sess.client('data')

        search_filter = planet.data_filter.and_filter([
            planet.data_filter.geometry_filter(AOI_GEOJSON),
            planet.data_filter.date_range_filter(
                'acquired',
                gte=datetime.strptime(START_DATE, '%Y-%m-%d'),
                lte=datetime.strptime(END_DATE,   '%Y-%m-%d')
            ),
            planet.data_filter.range_filter('cloud_cover', lte=MAX_CLOUD),
            planet.data_filter.asset_filter([ASSET_TYPE])
        ])

        items = []
        async for item in client.search([ITEM_TYPE], search_filter, limit=MAX_SCENES):
            items.append(item)

        return items

# Run the search
scenes = asyncio.run(search_scenes())
scene_ids = [s['id'] for s in scenes]

print(f'✅ Found {len(scenes)} scenes')
if scenes:
    dates = sorted([s['properties']['acquired'][:10] for s in scenes])
    print(f'   Earliest: {dates[0]}')
    print(f'   Latest:   {dates[-1]}')
    print(f'   First 5 scene IDs:')
    for sid in scene_ids[:5]:
        print(f'     {sid}')
else:
    print('❌ No scenes found — check AOI, date range, or cloud filter')

✅ Found 5 scenes
   Earliest: 2021-05-08
   Latest:   2021-12-05
   First 5 scene IDs:
     20210508_031940_25_2262
     20211127_030838_51_2420
     20211205_030503_29_241a
     20211205_030500_83_241a
     20211205_030458_36_241a


---
## Cell 6 — 📦 Place Planet Order
> Planet will deliver scenes directly to your GCS bucket

In [ ]:
# async def place_order(ids, order_name='mining_site_order'):
#     async with planet.Session(auth=planet.Auth.from_key(PLANET_API_KEY)) as sess:
#         client = sess.client('orders')

#         # Read service account credentials for GCS delivery
#         with open(GCS_CREDENTIALS, 'r') as f:
#             creds_json = f.read()

#         delivery = planet.order_request.google_cloud_storage_delivery(
#             bucket=GCS_BUCKET,
#             credentials=creds_json,
#             path_prefix=GCS_PREFIX
#         )

#         request = planet.order_request.build_request(
#             name=order_name,
#             products=[
#                 planet.order_request.product(
#                     item_ids=ids,
#                     product_bundle=BUNDLE_TYPE,
#                     item_type=ITEM_TYPE
#                 )
#             ],
#             delivery=delivery,
#             tools=[planet.order_request.clip_tool(AOI_GEOJSON)]  # clip to AOI
#         )

#         order = await client.create_order(request)
#         return order

# # Place the order
# order = asyncio.run(place_order(scene_ids))
# ORDER_ID = order['id']

# print(f'✅ Order placed successfully!')
# print(f'   Order ID: {ORDER_ID}')
# print(f'   Status:   {order["state"]}')
# print(f'   Scenes:   {len(scene_ids)}')
# print(f'\n💾 Save your Order ID in case you need to resume:')
# print(f'   ORDER_ID = "{ORDER_ID}"')

In [15]:
# ── Quick GCS write test ───────────────────────────────────────────────
from google.cloud import storage

gcs    = storage.Client.from_service_account_json(GCS_CREDENTIALS)
bucket = gcs.bucket(GCS_BUCKET)

# Try writing a test file
blob = bucket.blob('test_write_access.txt')
blob.upload_from_string('write test ok')
blob.delete()
print('✅ GCS write/delete access confirmed')

✅ GCS write/delete access confirmed


In [ ]:
import base64
import json

async def place_order(ids, order_name='mining_site_order'):
    async with planet.Session(auth=planet.Auth.from_key(PLANET_API_KEY)) as sess:
        client = sess.client('orders')

        with open(GCS_CREDENTIALS, 'rb') as f:
            creds_b64 = base64.b64encode(f.read()).decode('utf-8')

        request = {
            "name": order_name,
            "products": [
                {
                    "item_ids": ids,
                    "item_type": ITEM_TYPE,
                    "product_bundle": BUNDLE_TYPE
                }
            ],
            "delivery": {
                "google_cloud_storage": {
                    "bucket": GCS_BUCKET,
                    "credentials": creds_b64,
                    "path_prefix": GCS_PREFIX
                }
            }
            # ← no tools at all for now
        }

        order = await client.create_order(request)
        return order

# Test with 5 scenes
order = asyncio.run(place_order(scene_ids[:5]))
ORDER_ID = order['id']
print(f'✅ Order placed: {ORDER_ID}')
print(f'   Status: {order["state"]}')

✅ Order placed: f168f0a0-e991-4230-859f-4c9e9cee543b
   Status: queued


---
## Cell 7 — ⏳ Monitor Order Status
> Polls every 60 seconds. Large orders can take 30–60 minutes.
> 
> If the kernel disconnects, use **Cell 7b** to resume with a saved Order ID.

In [17]:
async def wait_for_order(order_id, poll_interval=60):
    async with planet.Session(auth=planet.Auth.from_key(PLANET_API_KEY)) as sess:
        client = sess.client('orders')
        attempt = 0

        while True:
            attempt += 1
            order = await client.get_order(order_id)
            state = order['state']
            ts    = datetime.now().strftime('%H:%M:%S')
            print(f'[{ts}] Attempt {attempt:>3} — Status: {state}')

            if state == 'success':
                print('\n✅ Order complete! Files are in your GCS bucket.')
                return order
            elif state in ('failed', 'partial'):
                print(f'\n⚠️  Order ended with state: {state}')
                return order
            elif state == 'cancelled':
                raise RuntimeError('Order was cancelled')

            await asyncio.sleep(poll_interval)

completed_order = asyncio.run(wait_for_order(ORDER_ID))

[17:33:07] Attempt   1 — Status: running
[17:34:07] Attempt   2 — Status: running
[17:35:07] Attempt   3 — Status: running
[17:36:07] Attempt   4 — Status: running
[17:37:08] Attempt   5 — Status: running
[17:38:08] Attempt   6 — Status: running
[17:39:08] Attempt   7 — Status: running
[17:40:08] Attempt   8 — Status: running
[17:41:09] Attempt   9 — Status: running
[17:42:09] Attempt  10 — Status: running
[17:43:09] Attempt  11 — Status: running
[17:44:09] Attempt  12 — Status: running
[17:45:09] Attempt  13 — Status: running
[17:46:10] Attempt  14 — Status: running
[17:47:10] Attempt  15 — Status: running
[17:48:10] Attempt  16 — Status: running
[17:49:10] Attempt  17 — Status: running
[17:50:11] Attempt  18 — Status: running
[17:51:11] Attempt  19 — Status: running
[17:52:11] Attempt  20 — Status: running
[17:53:11] Attempt  21 — Status: success

✅ Order complete! Files are in your GCS bucket.


In [ ]:
# ── Cell 7b — Resume monitoring with a saved Order ID ──────────────────
# Run this cell if your kernel restarted and you need to re-check an order

# ORDER_ID = 'paste-your-order-id-here'   # ← uncomment and paste ID
# completed_order = asyncio.run(wait_for_order(ORDER_ID))

---
## Cell 8 — 📁 List GeoTIFFs in GCS

In [18]:
def list_gcs_tifs():
    gcs    = storage.Client.from_service_account_json(GCS_CREDENTIALS)
    bucket = gcs.bucket(GCS_BUCKET)
    blobs  = list(bucket.list_blobs(prefix=GCS_PREFIX))
    tifs   = [b for b in blobs if b.name.endswith('.tif') and 'SR' in b.name]
    return tifs

tif_blobs = list_gcs_tifs()

print(f'✅ Found {len(tif_blobs)} GeoTIFFs in GCS')
print('\nFirst 5 files:')
for b in tif_blobs[:5]:
    print(f'  gs://{GCS_BUCKET}/{b.name}  ({b.size / 1e6:.1f} MB)')

✅ Found 5 GeoTIFFs in GCS

First 5 files:
  gs://planet-mining/planet-mining-site/f168f0a0-e991-4230-859f-4c9e9cee543b/PSScene/20210508_031940_25_2262_3B_AnalyticMS_SR.tif  (521.3 MB)
  gs://planet-mining/planet-mining-site/f168f0a0-e991-4230-859f-4c9e9cee543b/PSScene/20211127_030838_51_2420_3B_AnalyticMS_SR.tif  (629.9 MB)
  gs://planet-mining/planet-mining-site/f168f0a0-e991-4230-859f-4c9e9cee543b/PSScene/20211205_030458_36_241a_3B_AnalyticMS_SR.tif  (619.4 MB)
  gs://planet-mining/planet-mining-site/f168f0a0-e991-4230-859f-4c9e9cee543b/PSScene/20211205_030500_83_241a_3B_AnalyticMS_SR.tif  (623.5 MB)
  gs://planet-mining/planet-mining-site/f168f0a0-e991-4230-859f-4c9e9cee543b/PSScene/20211205_030503_29_241a_3B_AnalyticMS_SR.tif  (623.3 MB)


---
## Cell 9 — 🌍 Create GEE ImageCollection Asset

In [19]:
def create_gee_collection():
    try:
        ee.data.createAsset({'type': 'ImageCollection'}, GEE_ASSET_PATH)
        print(f'✅ Created GEE ImageCollection: {GEE_ASSET_PATH}')
    except Exception as e:
        if 'already exists' in str(e).lower():
            print(f'ℹ️  Collection already exists: {GEE_ASSET_PATH}')
        else:
            raise

create_gee_collection()

✅ Created GEE ImageCollection: projects/gee-planet-natcap/assets/planet_mining


---
## Cell 10 — ⬆️ Batch Ingest GCS → GEE

In [21]:
def ingest_tif_to_gee(blob, index):
    gcs_uri  = f'gs://{GCS_BUCKET}/{blob.name}'
    parts    = blob.name.split('/')

    # Extract filename e.g. 20210508_031940_25_2262_3B_AnalyticMS_SR.tif
    filename = parts[-1].replace('.tif', '')
    scene_id = f'scene_{index:04d}_{filename}'[:100]  # keep under 100 chars
    asset_id = f'{GEE_ASSET_PATH}/{scene_id}'

    # Parse date from filename (YYYYMMDD_...)
    try:
        date_str   = parts[-1][:8]          # YYYYMMDD from filename
        scene_date = f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}'
        timestamp  = int(datetime.strptime(scene_date, '%Y-%m-%d').timestamp() * 1000)
    except Exception:
        scene_date = START_DATE
        timestamp  = int(datetime.strptime(START_DATE, '%Y-%m-%d').timestamp() * 1000)

    params = {
        'id': asset_id,
        'tilesets': [{'sources': [{'uris': [gcs_uri]}]}],
        'bands': [
            {'id': 'B', 'tileset_band_index': 0},
            {'id': 'G', 'tileset_band_index': 1},
            {'id': 'R', 'tileset_band_index': 2},
            {'id': 'N', 'tileset_band_index': 3},
        ],
        'start_time': f'{scene_date}T00:00:00Z',   # ← top-level, NOT in properties
        'properties': {
            'scene_id': scene_id,
            'acquired': scene_date,
            'source': 'PlanetScope'
        }
    }

    task_id = ee.data.newTaskId()[0]
    ee.data.startIngestion(task_id, params)
    return task_id, scene_date


def batch_ingest(blobs):
    task_ids = []
    total    = len(blobs)

    for i, blob in enumerate(blobs):
        try:
            task_id, scene_date = ingest_tif_to_gee(blob, i)
            task_ids.append(task_id)
            print(f'  [{i+1:>4}/{total}] {scene_date} → task {task_id}')
            time.sleep(0.5)  # avoid GEE rate limit
        except Exception as e:
            print(f'  [{i+1:>4}/{total}] ❌ Failed: {blob.name} — {e}')

    return task_ids


print(f'⬆️  Starting ingestion of {len(tif_blobs)} scenes...')
task_ids = batch_ingest(tif_blobs)

print(f'\n✅ Submitted {len(task_ids)} ingestion tasks')
print('   Monitor at: https://code.earthengine.google.com/tasks')

⬆️  Starting ingestion of 5 scenes...
  [   1/5] 2021-05-08 → task 8c6bfed0-805f-422e-bbdb-90f40ed5e617
  [   2/5] 2021-11-27 → task 25c9a64f-2f24-41d9-99b7-25520e1ec0e7
  [   3/5] 2021-12-05 → task 69989b9c-70a4-4508-94a2-4d5da0777e6d
  [   4/5] 2021-12-05 → task faed1695-1301-48ed-a893-aea63f21469d
  [   5/5] 2021-12-05 → task 95d454ad-4d77-4895-9c23-3cb3253688f5

✅ Submitted 5 ingestion tasks
   Monitor at: https://code.earthengine.google.com/tasks


---
## Cell 11 — 📊 Check GEE Task Status

In [22]:
def check_task_status(task_ids, show_n=10):
    """Check status of submitted GEE ingestion tasks."""
    statuses = {'COMPLETED': 0, 'RUNNING': 0, 'PENDING': 0, 'FAILED': 0}
    failed   = []

    for tid in task_ids:
        try:
            status = ee.data.getTaskStatus(tid)[0]
            state  = status.get('state', 'UNKNOWN')
            if state in statuses:
                statuses[state] += 1
            if state == 'FAILED':
                failed.append((tid, status.get('error_message', 'Unknown error')))
        except Exception as e:
            print(f'  Could not check task {tid}: {e}')

    print('📊 Task Status Summary:')
    for state, count in statuses.items():
        emoji = {'COMPLETED':'✅','RUNNING':'⏳','PENDING':'🕐','FAILED':'❌'}[state]
        print(f'   {emoji} {state}: {count}')

    if failed:
        print('\n❌ Failed tasks:')
        for tid, err in failed[:show_n]:
            print(f'   {tid}: {err}')

check_task_status(task_ids)

📊 Task Status Summary:
   ✅ COMPLETED: 0
   ⏳ RUNNING: 0
   🕐 PENDING: 0
   ❌ FAILED: 0


---
## Cell 12 — 💾 Save GEE Visualisation Script
> Paste the output `.js` file into [code.earthengine.google.com](https://code.earthengine.google.com)

In [ ]:
gee_script = f"""
// ═══════════════════════════════════════════════════════════════════
// Planet Mining Site — Land Use Change Viewer
// Auto-generated by planet_gee_pipeline.ipynb
// ═══════════════════════════════════════════════════════════════════

var AOI = ee.Geometry.Polygon({json.dumps(AOI_GEOJSON['coordinates'])});

var col = ee.ImageCollection('{GEE_ASSET_PATH}')
  .filterBounds(AOI)
  .filterDate('{START_DATE}', '{END_DATE}')
  .sort('system:time_start');

print('Total scenes:', col.size());

var VIS_RGB = {{bands: ['R', 'G', 'B'], min: 0, max: 3000, gamma: 1.4}};
var VIS_NIR = {{bands: ['N', 'R', 'G'], min: 0, max: 3000, gamma: 1.4}};

// ── Annual mosaics ─────────────────────────────────────────────────
var years = ee.List.sequence({START_DATE[:4]}, {END_DATE[:4]});
var annualMosaics = ee.ImageCollection(years.map(function(y) {{
  var start = ee.Date.fromYMD(y, 1, 1);
  return col.filterDate(start, start.advance(1, 'year'))
            .median().clip(AOI)
            .set('year', y)
            .set('system:time_start', start.millis());
}}));

// ── Map display ────────────────────────────────────────────────────
Map.centerObject(AOI, 13);
Map.setOptions('SATELLITE');
Map.addLayer(annualMosaics.sort('year').first(), VIS_RGB, 'Earliest RGB');
Map.addLayer(annualMosaics.sort('year', false).first(), VIS_RGB, 'Latest RGB');
Map.addLayer(annualMosaics.sort('year', false).first(), VIS_NIR, 'Latest False Colour');

// ── Side-by-side split panel ───────────────────────────────────────
var leftMap  = ui.Map(); leftMap.setOptions('SATELLITE');
var rightMap = ui.Map(); rightMap.setOptions('SATELLITE');
ui.Map.Linker([leftMap, rightMap]);
leftMap.centerObject(AOI, 13);

var years_list = [];
for (var y = {START_DATE[:4]}; y <= {END_DATE[:4]}; y++) years_list.push(String(y));

var leftSel  = ui.Select({{items: years_list, value: '{START_DATE[:4]}',
  onChange: function(v) {{
    leftMap.layers().reset();
    leftMap.addLayer(
      annualMosaics.filter(ee.Filter.eq('year', parseInt(v))).first(), VIS_RGB, 'Left: '+v);
  }}}});
var rightSel = ui.Select({{items: years_list, value: '{END_DATE[:4]}',
  onChange: function(v) {{
    rightMap.layers().reset();
    rightMap.addLayer(
      annualMosaics.filter(ee.Filter.eq('year', parseInt(v))).first(), VIS_RGB, 'Right: '+v);
  }}}});

leftMap.addLayer(annualMosaics.sort('year').first(), VIS_RGB, 'Left: {START_DATE[:4]}');
rightMap.addLayer(annualMosaics.sort('year', false).first(), VIS_RGB, 'Right: {END_DATE[:4]}');

var controls = ui.Panel([  
  ui.Label('🛰 Planet Mining Site Change Viewer', {{fontWeight:'bold', fontSize:'14px'}}),
  ui.Panel([ui.Label('◀ Left year:'), leftSel],  ui.Panel.Layout.flow('horizontal')),
  ui.Panel([ui.Label('Right year: ▶'), rightSel], ui.Panel.Layout.flow('horizontal'))
], null, {{position:'top-center', backgroundColor:'rgba(255,255,255,0.9)', padding:'8px'}});

var splitPanel = ui.SplitPanel({{firstPanel: leftMap, secondPanel: rightMap, wipe: true}});
ui.root.clear();
ui.root.add(ui.Panel([splitPanel, controls], ui.Panel.Layout.flow('vertical'), {{stretch:'both'}}));

// ── Animated GIF ───────────────────────────────────────────────────
print('🎞 GIF URL:', annualMosaics.select(['R','G','B']).getVideoThumbURL({{
  region: AOI, dimensions: 512, framesPerSecond: 2,
  min: 0, max: 3000, gamma: 1.4, format: 'GIF'
}}));

// ── Export to Drive ────────────────────────────────────────────────
Export.image.toDrive({{
  image: annualMosaics.sort('year', false).first().select(['R','G','B']).toFloat(),
  description: 'Planet_mining_latest',
  folder: 'GEE_mining_planet',
  region: AOI,
  scale: 4.77,
  maxPixels: 1e13,
  fileFormat: 'GeoTIFF',
  formatOptions: {{ noData: -9999.0 }}
}});
"""

script_path = Path('./gee_mining_visualisation.js')
script_path.write_text(gee_script)

print(f'✅ GEE script saved to: {script_path.resolve()}')
print('\n👉 Next steps:')
print('   1. Open https://code.earthengine.google.com')
print('   2. Paste the contents of gee_mining_visualisation.js')
print('   3. Click Run')

---
## Cell 13 — 🧹 Optional: Delete GCS Files After Ingest
> Run this only after confirming all GEE tasks completed successfully

In [ ]:
# Uncomment to delete GCS files and save storage costs after ingest is confirmed

# def cleanup_gcs(blobs):
#     gcs    = storage.Client.from_service_account_json(GCS_CREDENTIALS)
#     bucket = gcs.bucket(GCS_BUCKET)
#     for blob in blobs:
#         bucket.blob(blob.name).delete()
#         print(f'  🗑️  Deleted: {blob.name}')
#     print(f'✅ Deleted {len(blobs)} files from GCS')

# cleanup_gcs(tif_blobs)